In [ ]:
%load_ext autoreload
%autoreload 2 

## Imports

In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_gpu_deterministic_ops=true"

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import flax
import pandas as pd
from scipy.signal import detrend
from statsmodels.tsa.filters.hp_filter import hpfilter
from scipy import optimize
import sys

sys.path.append("py-files")

In [ ]:
do_64_bit = False
if do_64_bit: jax.config.update("jax_enable_x64", True)

In [ ]:
import solve
import aux_ as aux
import RANK
import model_funcs
import neural_nets
import plotting

# Data

In [ ]:
AWMD_df = pd.read_excel('AWMD/AWMD_Mar2026.xlsx', sheet_name=2)

# drop latest observations (it is NaN)
AWMD_df = AWMD_df[:-1]
AWMD_df = AWMD_df.set_index(AWMD_df['date']).drop("date", axis=1)

indx_df = AWMD_df.index

In [ ]:
Y_raw = AWMD_df["YER"]
P_raw = AWMD_df["YED"]
i_raw = AWMD_df["STN"]

In [ ]:
Y_cycle, Y_trend = hpfilter(Y_raw, lamb=1600)
Y_dev = Y_cycle/Y_trend

pi_raw = P_raw.pct_change(1).dropna() # drop 1st obs, since it is NaN from pct_change function
pi = detrend(pi_raw)
pi = pd.Series(pi)

i = detrend(i_raw)
i = pd.Series(i)

In [ ]:
f, ax = plt.subplots(2,3,figsize=(15,10))

ax[0,0].plot(indx_df, 100*Y_dev)
ax[0,1].plot(indx_df[1:], 100*pi)
ax[0,2].plot(indx_df, i)

ax[1,0].hist(100*Y_dev, bins=20)
ax[1,1].hist(100*pi, bins=20)
ax[1,2].hist(i, bins=20)

for j in range(2):
    ax[j, 0].set_title('Output (HP-filtered)')
    ax[j, 1].set_title('Quarterly GDP-inflation')
    ax[j, 2].set_title('Short Interest Rate')

In [ ]:
Y_dev_var = Y_dev.var()
pi_var = pi.var()
i_var = i.var()

Y_dev_ac1 = Y_dev.autocorr(lag=1).item()
pi_ac1 = pi.autocorr(lag=1).item()
i_ac1 = i.autocorr(lag=1).item()

corr_df = pd.DataFrame({
    'Y_dev': Y_dev.iloc[1:].reset_index(drop=True),
    'pi': pi.reset_index(drop=True),
    'i': i.iloc[1:].reset_index(drop=True)
}).corr()

Y_dev_pi_corr = corr_df.iloc[0,1]
Y_dev_i_corr = corr_df.iloc[0,2]
pi_i_corr = corr_df.iloc[1,2]

summary_df = pd.DataFrame({
    'variable': ['Y_dev', 'pi', 'i'],
    'variance': [Y_dev_var, pi_var, i_var],
    'ac1': [Y_dev_ac1, pi_ac1, i_ac1]
})

print(summary_df)
print("\nCorrelations:")
print(corr_df)


In [ ]:
true_moments = np.array([Y_dev_var, Y_dev_ac1, pi_var, pi_ac1])

## Device

In [ ]:
device = aux.choose_gpu()

# Estimation

In [ ]:
model = RANK.RANK_model(device)

In [ ]:
def update_par(model, new_par):

    for k,v in new_par.items():
        model.par[k] = v

    return model

def compute_moments(series):

    df_ = pd.DataFrame(series)
    var = df_.var(axis=0).mean()
    ac1_ = df_.apply(lambda col: col.autocorr(lag=1), axis=0)
    ac1 = ac1_.mean()

    return var, ac1

def compute_corr(series1, series2):
    return pd.DataFrame(series1).corrwith(pd.DataFrame(series2), axis=0).mean()

In [ ]:
sigma_eps_u = 0.00135
rho_u = 0.84

# initial guesses
sigma_eps_z = 0.01
sigma_eps_Gamma = 0.01

rho_z = 0.9
rho_Gamma = 0.9

new_par = {"rho_u" : rho_u, "rho_z" : rho_z, "rho_Gamma" : rho_Gamma}
model = update_par(model, new_par)

sigma_quad = {
    "sigma_eps_u" : sigma_eps_u,
    "sigma_eps_z" :  sigma_eps_z,
    "sigma_eps_Gamma" : sigma_eps_Gamma
}

factor = 1

sigma_sim = {
    "sigma_eps_u" : factor*sigma_eps_u,
    "sigma_eps_z" :  factor*sigma_eps_z,
    "sigma_eps_Gamma" : factor*sigma_eps_Gamma
}

episode_list = [100000]
lr_list = [1e-4]
N_list = [1000]
ZLB_list = [-100]

In [ ]:
def obj(x, factor=1, Tsim=100_000, Nsim=500, do_print=True):

    # x is (4,) np.array, unpack
    rho_z_i = x[0]
    rho_Gamma_i = x[1]
    sigma_eps_z_i = x[2]
    sigma_eps_Gamma_i = x[3]

    # update model
    new_par = {"rho_z" : rho_z_i, "rho_Gamma" : rho_Gamma_i}
    sigma_quad["sigma_eps_z"] = sigma_eps_z_i
    sigma_quad["sigma_eps_Gamma"] = sigma_eps_Gamma_i
    sigma_sim["sigma_eps_z"] = factor*sigma_eps_z_i
    sigma_sim["sigma_eps_Gamma"] = factor*sigma_eps_Gamma_i

    if do_print:
        print('\n\n')
        print(f'rho_z = \t\t{rho_z_i:.6f}')
        print(f'rho_Gamma = \t\t{rho_Gamma_i:.6f}')
        print(f'sigma_eps_z = \t\t{sigma_eps_z_i:.6f}')
        print(f'sigma_eps_Gamma = \t{sigma_eps_Gamma_i:.6f}')

    model_i = update_par(model, new_par)

    # solve new model
    solve.train_nn(model_i, episode_list, sigma_sim, sigma_quad, ZLB_list, lr_list, N_list, print_freq=100, early_termination=True)

    # simulate model
    solve.simulate(model_i, Nsim, Tsim, sigma_quad, ZLB=-100, do_lin=False, do_OccBin=False)

    # unpack
    Y_dev_i = model_i.sim.Y_dev
    pi_i = model_i.sim.pi

    # compute moments
    Y_dev_var_i, Y_dev_ac1_i = compute_moments(Y_dev_i)
    pi_var_i, pi_ac1_i = compute_moments(pi_i)

    # stack in error array
    model_moments = np.array([Y_dev_var_i, Y_dev_ac1_i, pi_var_i, pi_ac1_i])

    # compute per. deviations of model moments from true moments
    errors = (model_moments - true_moments)/true_moments

    # square
    errors_sq = errors**2

    # scalar
    errors_scalar = np.sum(errors_sq)

    if do_print: print(errors)

    return errors_scalar

In [ ]:
# initial guess
x0 = np.array([0.9, 0.9, 0.01, 0.01])
bounds = [
    (0.00, 0.999),
    (0.00, 0.999),
    (0.00, 0.05),
    (0.00, 0.05)
]

In [ ]:
res = optimize.minimize(obj, x0, bounds=bounds, method="Powell")

In [ ]:
res

# Comparing the estimated model with the AWM data

In [ ]:
x = res.x

In [ ]:
rho_z = x[0]
rho_Gamma = x[1]
sigma_eps_z = x[2]
sigma_eps_Gamma = x[3]

# update model
new_par = {"rho_z" : rho_z, "rho_Gamma" : rho_Gamma}
sigma_quad["sigma_eps_z"] = sigma_eps_z
sigma_quad["sigma_eps_Gamma"] = sigma_eps_Gamma

print(f'rho_z = \t\t{rho_z:.6f}')
print(f'rho_Gamma = \t\t{rho_Gamma:.6f}')
print(f'sigma_eps_z = \t\t{sigma_eps_z:.6f}')
print(f'sigma_eps_Gamma = \t{sigma_eps_Gamma:.6f}')

model = update_par(model, new_par)

In [ ]:
solve.train_nn(model, episode_list, sigma_sim, sigma_quad, ZLB_list, lr_list, N_list, zero_var_list, print_freq=100, early_termination=True)

In [ ]:
Nsim = 500
Tsim = 10000

solve.simulate(model, Nsim, Tsim, sigma_quad, ZLB=-100, do_lin=True, do_OccBin=False)

In [ ]:
plt.rcParams.update({'font.size': 20})

f, ax = plt.subplots(1,3,figsize=(15,5))

ax[0].hist(100*Y_dev, density=True, bins=50, label='AWM data')
ax[0].hist(100*model.sim.Y_dev.flatten(), bins=50, density=True, alpha=0.7, label='DEQN w/o ZLB')
ax[0].set_xlabel('% deviation from trend/SSS')
ax[0].set_title('Output')

ax[1].hist(100*pi, density=True, bins=50)
ax[1].hist(100*model.sim.pi_dev.flatten(), bins=50, density=True, alpha=0.7)
ax[1].set_xlabel('p.p. deviation from trend/SSS')
ax[1].set_title('Inflation')

ax[2].hist(i, density=True, bins=50)
ax[2].hist(100*model.sim.i_dev.flatten(), bins=50, density=True, alpha=0.7)
ax[2].set_xlabel('p.p. deviation from trend/SSS')
ax[2].set_title('Short Interest Rate')

f.legend(loc='lower center', ncols=2, bbox_to_anchor=(0.5, -0.04))

f.tight_layout()

f.savefig('plots/Fig2_AWM_vs_DEQNwoZLB.png')

In [ ]:
Y_dev_var_model, Y_dev_ac1_model = compute_moments(model.sim.Y_dev)
pi_dev_var_model, pi_dev_ac1_model = compute_moments(model.sim.pi_dev)
i_dev_var_model, i_dev_ac1_model = compute_moments(model.sim.i_dev)

Y_dev_pi_corr_model = compute_corr(model.sim.Y_dev, model.sim.pi_dev)
Y_dev_i_corr_model = compute_corr(model.sim.Y_dev, model.sim.i_dev)
pi_i_corr_model = compute_corr(model.sim.pi_dev, model.sim.i_dev)

In [ ]:
moments = {
    "Y_dev Variance": (10000*Y_dev_var, 10000*Y_dev_var_model),
    "Y_dev AC1": (Y_dev_ac1, Y_dev_ac1_model),
    "pi_dev Variance": (10000*pi_var, 10000*pi_dev_var_model),
    "pi_dev AC1": (pi_ac1, pi_dev_ac1_model),
    "i_dev Variance": (i_var, 10000*i_dev_var_model),
    "i_dev AC1": (i_ac1, i_dev_ac1_model),
    "Y_dev / pi_dev Corr": (Y_dev_pi_corr, Y_dev_pi_corr_model),
    "Y_dev / i_dev Corr": (Y_dev_i_corr, Y_dev_i_corr_model),
    "pi_dev / i_dev Corr": (pi_i_corr, pi_i_corr_model)
}

for name, (data, model_val) in moments.items():
    print(f"{name:20} | Data: {data:10.5f} | Model: {model_val:10.5f}")